In [1]:
from sklearn.metrics import ndcg_score
import numpy as np
from collections import Counter
import pandas as pd

def calculate_relevance_metrics(true_relevance, predicted_scores):
    return {
        'NDCG@10': ndcg_score([true_relevance], [predicted_scores], k=10),
        'Precision@10': len(set(true_relevance[:10]) & set(predicted_scores[:10]))/10,
        'Recall@10': len(set(true_relevance) & set(predicted_scores[:10]))/len(true_relevance)
    }    

In [ ]:
# def calculate_diversity_metrics(recommendations):
#     categories = [cat for rec in recommendations for cat in rec.get('categories', [])]
#     category_counts = Counter(categories)

#     return {
#         'CategoryCoverage': len(category_counts) / len(recommendations),
#         'Entropy': -sum((count/len(recommendations)) * np.log2(count/len(recommendations)) 
#                   for count in category_counts.values()),
#         'GiniIndex': 1 - sum((count/len(recommendations))**2 for count in category_counts.values())
#     }

In [ ]:
def clean_categories(categories_list):
    if not isinstance(categories_list, list):
        return []
    
    generic_categories = {'CDs & Vinyl', 'Digital Music', 'Music', 'Albums'}
    
    cleaned = []
    for category in categories_list:
        if isinstance(category, str):
            if category not in generic_categories:
                cleaned.append(category)
        elif isinstance(category, list):
            cleaned.extend([c for c in category if c not in generic_categories])
    
    return list(set(cleaned)) 

def calculate_diversity_metrics(recommendations):
    categories = []
    for rec in recommendations:
        cleaned_cats = clean_categories(rec.get('categories', []))
        categories.extend(cleaned_cats)
    
    category_counts = Counter(categories)
    
    return {
        'CategoryCoverage': len(category_counts) / len(recommendations) if recommendations else 0,
        'UniqueCategories': len(category_counts),
        'Entropy': -sum((count/len(categories)) * np.log2(count/len(categories)) 
                      for count in category_counts.values()) if categories else 0,
        'GiniIndex': 1 - sum((count/len(categories))**2 for count in category_counts.values()) if categories else 0
    }

In [ ]:
def calculate_novelty_metrics(recommendations, user_history):
    seen_items = set(user_history)
    return {
        'Novelty@10': len([rec for rec in recommendations[:10] if rec['iid'] not in seen_items]) / 10,
        'Serendipity@10': len([rec for rec in recommendations[:10] 
                             if rec['iid'] not in seen_items and rec['hybrid_score'] > 0.5]) / 10
    }

In [ ]:
import pandas as pd
from tqdm import tqdm

def run_benchmark(test_queries, user_history, n_runs=10):
    results = []
    
    for query, true_relevance in tqdm(test_queries.items()):
        run_metrics = []
        for _ in range(n_runs):
            # Get recommendations
            recommendations = full_pipeline("test_user", query, debug=False)
            
            # Calculate metrics
            metrics = {
                **calculate_relevance_metrics(true_relevance, [r['iid'] for r in recommendations]),
                **calculate_diversity_metrics(recommendations),
                **calculate_novelty_metrics(recommendations, user_history)
            }
            run_metrics.append(metrics)
        
        # aggregate results
        avg_metrics = pd.DataFrame(run_metrics).mean().to_dict()
        results.append({'query': query, **avg_metrics})
    
    return pd.DataFrame(results)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def intra_list_diversity(recommendations, embedder):
    """Calculate average dissimilarity between recommended items"""
    embeddings = [embedder.embed_query(rec['augmented_text']) for rec in recommendations[:10]]
    similarity_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(similarity_matrix, 0)  # Ignore self-similarity
    return 1 - similarity_matrix.mean()

In [ ]:
def mmr_effectiveness(recommendations):
    """Compare diversity before/after MMR"""
    pre_mmr_scores = [r['hybrid_score'] for r in recommendations]
    post_mmr_scores = [r['hybrid_score']*(1-r['mmr_score']) for r in recommendations]
    return {
        'MMR_Impact': np.mean(post_mmr_scores) - np.mean(pre_mmr_scores),
        'Diversity_Gain': calculate_diversity_metrics(recommendations)['CategoryCoverage']
    }

In [ ]:
def evaluate_pipeline():
    # 1. Run benchmark
    benchmark_results = run_benchmark(test_queries, user_history)
    
    # 2. Calculate aggregate metrics
    aggregate_metrics = {
        'Mean_NDCG@10': benchmark_results['NDCG@10'].mean(),
        'Mean_CategoryCoverage': benchmark_results['CategoryCoverage'].mean(),
        'Mean_Novelty@10': benchmark_results['Novelty@10'].mean(),
        'Diversity_Stability': benchmark_results['Entropy'].std()  # Lower is better
    }
    
    # 3. Specialized diversity evaluation
    sample_recs = full_pipeline("test_user", "relaxing jazz piano")
    diversity_metrics = {
        **intra_list_diversity(sample_recs, embedder),
        **mmr_effectiveness(sample_recs)
    }
    
    return {
        'benchmark_results': benchmark_results,
        'aggregate_metrics': aggregate_metrics,
        'diversity_metrics': diversity_metrics
    }

In [ ]:
import matplotlib.pyplot as plt

def plot_metrics(results):
    fig, ax = plt.subplots(2, 2, figsize=(15, 10))
    
    # Relevance metrics
    results['benchmark_results'][['NDCG@10', 'Precision@10']].plot(kind='bar', ax=ax[0,0])
    ax[0,0].set_title('Relevance Metrics')
    
    # Diversity metrics
    results['benchmark_results'][['CategoryCoverage', 'Entropy']].plot(kind='bar', ax=ax[0,1])
    ax[0,1].set_title('Diversity Metrics')
    
    # Novelty metrics
    results['benchmark_results'][['Novelty@10', 'Serendipity@10']].plot(kind='bar', ax=ax[1,0])
    ax[1,0].set_title('Novelty Metrics')
    
    # MMR impact
    pd.DataFrame([results['diversity_metrics']]).plot(kind='bar', ax=ax[1,1])
    ax[1,1].set_title('MMR Effectiveness')
    
    plt.tight_layout()
    plt.show()